# Geoloc



## infos
Participants : Hélène DALON-DENEE, Dame DIENG, Célien GRIL, Nathan LEBRE

ligne 42368 : photo id 5464485473, correction -> les dates étaient décalée, le # de minutes (25) était collé au titre de la photo ("une lundi matin comme tout les autre ;-(") et le décalage était propagé.

In [2]:
import numpy as np
import pandas as pd
import folium as fl
import hdbscan
import re

In [3]:
read_data = pd.read_csv("./flickr_data2.csv")

# Retirer les 142 lignes qui ont des valeurs non-nulles qui dépassent les colonnes attendues. 
# (unnamed 16, dont 2 lignes avec des valeurs dans unnamed 18)
wrong_data = read_data.dropna(how="all", subset=read_data.columns[[16, 18]])
fix_data = read_data.drop(index=wrong_data.index, axis=1)
toDropColumns = read_data.columns[[16, 17, 18]]
fix_data = fix_data.drop(toDropColumns, axis=1)

# Retirer les tuples dupliquée (ignore id et tags pour enlever certains dupliqué quand un post est modifié 
# et certains carousels, passe de 187544 à 175688 donc pas tant que ça, la majorité sont dupliqués point barre)
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " tags"]), keep='last')

# Retirer les carousels
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " title"]))

C:\Users\celie\AppData\Local\Temp\ipykernel_19920\2908680622.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  read_data = pd.read_csv("./flickr_data2.csv")


## Clustering

### HBDSCAN

In [4]:
data2D = fix_data[[' lat', ' long']]
coords_rad = np.radians(data2D)

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size = 100, min_samples = 50, metric= 'haversine')
labels = clusterer.fit_predict(coords_rad)

In [6]:
df = data2D.copy()
df["cluster"] = labels
pois = (
    df[df.cluster != -1]
    .groupby("cluster")
    .agg(
        lat_mean=(" lat", "mean"),
        lon_mean=(" long", "mean"),
        nb_photos=("cluster", "count")
    )
    .sort_values("nb_photos", ascending=False)
)

# Text mining

In [7]:
nona_df = fix_data[' tags'].dropna()

In [8]:
idf={}  # Inverse Document Frequency dictionary
total_docs = len(nona_df)
removed_tags=["cospla","japa","feminicide","girl","hair","overwatch","tracer","aplusphoto","view","fiume","night","chaise","chair","iphone","streetphotography","rue","gens","nuit","francia","poste","river","notte","night"]

for tags in nona_df:
    unique_tags = set(re.findall(r'[^\W\d_]{2,}', str(tags)))    
    unique_tags = {tag.lower() for tag in unique_tags if not any(bad in tag for bad in removed_tags)}  # Normalize to lowercase
    for tag in unique_tags:
        idf[tag] = idf.get(tag, 0) + 1

for tag in idf:
    idf[tag] = total_docs / idf[tag]

print(f"Number of tags: {len(idf)}")

Number of tags: 39857


In [ ]:
#Cleaning by the lower IDF values

threshold = 14  # Default threshold

idf2 = {tag:value for tag, value in idf.items() if value < threshold}
print(f"Number of tags after filtering with threshold {threshold}: {len(idf2)}")
#print("IDF values for tags:")
#for tag, value in idf2.items():
#    print(f"{tag}: {value}")

filtered_tags = [tag for tag, _ in idf2.items()]

Number of tags after filtering with threshold 14: 33


In [10]:
mining_data = df.copy()
tags_data = fix_data[" tags"]
mining_data = mining_data.merge(tags_data, left_index=True, right_index=True)
mining_data = mining_data.dropna(subset=[" tags"])

In [11]:
lonely_letters = ["a", "à", "â", "b", "c", "d", "e", "é", "è", "ê", "f", "g", "h", "i", "î", "j", "k", "l", "m", "n", "o", "ô", "p", "q", "r", "s", "t", "u", "û", "v", "w", "x", "y", "z"]
french_stop_words = ["ah", "ha", "aie", "au", "aux", "de", "des", "du", "le", "les", "un", "une", "ce", "ces", "se", "ses", "cet", "cette", "celui", "celle", "et", "ma", "mon", "mes"]
#nltk_eng_stop_words = ["me", "my", "we", "our", "ours", "you", "your", "yours", "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their", "theirs", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "an", "the", "and", "but", "if", "or", "as", "until", "of", "at", "by", "for", "with", "before", "after", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "then", "here", "there", "when", "where", "why", "how", "all", "any", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "can", "will", "don", "now"]

stop_words = lonely_letters + french_stop_words #+ nltk_eng_stop_words # il semble que les stop words anglais n'a
filtered_tags = filtered_tags + stop_words

In [12]:
unreduced_tags_dict = [{} for _ in range(301)]
for row in mining_data.itertuples():
    clust = getattr(row, 'cluster')
    tags = getattr(row, '_4')
    if (clust >= 0):
        tags = set(re.findall(r'[^\W\d_]+', str(tags)))
        for tag in tags:
            if tag in filtered_tags:
                continue
            unreduced_tags_dict[clust][tag] = unreduced_tags_dict[clust].get(tag, 0) + 1


In [26]:
percentile_parameter = 99
tags_dict = [{} for _ in range(301)]
for clust in range (301):
    values = [value for _, value in unreduced_tags_dict[clust].items()]
    if values == []:
        clust_threshold = -1
    else:
        clust_threshold = np.percentile(values, percentile_parameter)
    for tag in list(unreduced_tags_dict[clust].keys()):
        if unreduced_tags_dict[clust][tag] >= clust_threshold:
            tags_dict[clust][tag] = unreduced_tags_dict[clust][tag]

In [27]:
# Réinitialiser les indices pour éviter les problèmes
donnees_geoloc_reset = data2D.reset_index(drop=True)
labels_reset = labels.copy()

# Créer la carte centrée sur Lyon
map_hdbscan = fl.Map(
    location=[45.757778, 4.832222],
    zoom_start=12
)

# Couleurs pour les clusters
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
          'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'gray']

# Ajouter les centres des clusters (POIs)
for cluster_id, poi_row in pois.iterrows():
    fl.Marker(
        location=[poi_row['lat_mean'], poi_row['lon_mean']],
        popup=f"Cluster {cluster_id}<br>{int(poi_row['nb_photos'])} photos <br>Tags: {', '.join(tags_dict[cluster_id].keys())}",
        icon=fl.Icon(color=colors[int(cluster_id) % len(colors)], icon='info-sign')
    ).add_to(map_hdbscan)

# Sauvegarder la carte
map_hdbscan.save('map_tag_on.html')
print("Carte sauvegardée dans 'map_tag_on.html'")

Carte sauvegardée dans 'map_tag_on.html'
